In [3]:
import pandas as pd
from pathlib import Path
from pprint import pprint
from import_data import get_dataset

c:\Users\vinic\OneDrive\Desktop\geral\dashs\dash_entrevista\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Carregamento dos datasets

In [4]:
## carregamento co  função get_dataset

get_dataset()

In [5]:
DIR = Path('dados')

nomes = {
       'olist_customers_dataset.csv': 'dados_clientes',
       'olist_geolocation_dataset.csv': 'dados_geograficos',
       'olist_orders_dataset.csv': 'dados_pedidos',
       'olist_order_items_dataset.csv': 'dados_itens_pedidos',
       'olist_order_payments_dataset.csv': 'dados_pagamentos',
       'olist_order_reviews_dataset.csv': 'dados_avaliacoes',
       'olist_products_dataset.csv': 'dados_produtos',
       'olist_sellers_dataset.csv': 'dados_vendedores',
       'product_category_name_translation.csv': 'dados_nomes_categorias',
    }

dfs = {}

for dataset in DIR.iterdir():
    nome_dataset = nomes.get(dataset.name)
    if nome_dataset:
        dfs[nome_dataset] = pd.read_csv(dataset)

In [6]:
for df in dfs.keys():
    print(df)

dados_clientes
dados_geograficos
dados_pedidos
dados_itens_pedidos
dados_pagamentos
dados_avaliacoes
dados_produtos
dados_vendedores
dados_nomes_categorias


## Analise exploratória

In [7]:
## Quantidade de linhas e colunas em cada dataset

shape = {df_nome: df.shape for df_nome, df in dfs.items()}

for df_nome, df in dfs.items():
    print(f'{df_nome}: Linhas {df.shape[0]}, Colunas {df.shape[1]}')


dados_clientes: Linhas 99441, Colunas 5
dados_geograficos: Linhas 1000163, Colunas 5
dados_pedidos: Linhas 99441, Colunas 8
dados_itens_pedidos: Linhas 112650, Colunas 7
dados_pagamentos: Linhas 103886, Colunas 5
dados_avaliacoes: Linhas 99224, Colunas 7
dados_produtos: Linhas 32951, Colunas 9
dados_vendedores: Linhas 3095, Colunas 4
dados_nomes_categorias: Linhas 71, Colunas 2


In [8]:
## Valores Nulos

for df_nome, df in dfs.items():
    print(df_nome)
    print(df.isna().sum())
    print(30 * '-')

dados_clientes
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
------------------------------
dados_geograficos
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
------------------------------
dados_pedidos
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
------------------------------
dados_itens_pedidos
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: 

In [9]:
def calcula_nulos(coluna): 
    '''
    Retorna quantidade de dados Nulos, Total de dados e Percentual de uma coluna especifica
    '''
    qtd_nulos = coluna.isna().sum()
    qtd_total = coluna.shape[0]

    print('Quantidade de Produtos com categorias Nulas')
    print(f'Quantidade de Nulos: {qtd_nulos}')
    print(f'Quantidade Total: {qtd_total}')
    print(f'Percentual: {round(qtd_nulos / qtd_total, 3)}%')


### A 610 nomes de categorias nulas na tabela Produto

In [10]:
### 610 Nomes de categorias Nulas
df_produto = dfs['dados_produtos']
coluna = df_produto['product_category_name']

calcula_nulos(coluna)

Quantidade de Produtos com categorias Nulas
Quantidade de Nulos: 610
Quantidade Total: 32951
Percentual: 0.019%


In [11]:
## Renomeando para sem informação
df_produto['product_category_name'] = df_produto['product_category_name'].fillna('Sem Informação') 

### Na tabela dados pedidos existem dados de data de pagamento

In [12]:
df_pedidos = dfs['dados_pedidos']
coluna = df_pedidos['order_approved_at']

calcula_nulos(coluna)


Quantidade de Produtos com categorias Nulas
Quantidade de Nulos: 160
Quantidade Total: 99441
Percentual: 0.002%


In [13]:
## Todas as vendas com data de Pagamento Nulas são vendas canceladas

df_pedidos[df_pedidos['order_approved_at'].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaN,NaN,NaN,2018-09-12 00:00:00
1801,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16,NaN,NaN,NaN,2018-10-17 00:00:00
1868,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,canceled,2017-03-04 12:14:30,NaN,NaN,NaN,2017-04-10 00:00:00
2029,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,canceled,2018-08-29 16:27:49,NaN,NaN,NaN,2018-09-13 00:00:00
2161,7a9d4c7f9b068337875b95465330f2fc,7f71ae48074c0cfec9195f88fcbfac55,canceled,2017-05-01 16:12:39,NaN,NaN,NaN,2017-05-30 00:00:00
...,...,...,...,...,...,...,...,...
97696,5a00b4d35edffc56b825c3646a99ba9d,6a3bdf004ca96338fb5fad1b8d93c2e6,canceled,2017-07-02 15:38:46,NaN,NaN,NaN,2017-07-25 00:00:00
98415,227c804e2a44760671a6a5697ea549e4,62e7477e75e542243ee62a0ba73f410f,canceled,2017-09-28 15:02:56,NaN,NaN,NaN,2017-10-16 00:00:00
98909,e49e7ce1471b4693482d40c2bd3ad196,e4e7ab3f449aeb401f0216f86c2104db,canceled,2018-08-07 11:16:28,NaN,NaN,NaN,2018-08-10 00:00:00
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaN,NaN,NaN,2018-10-01 00:00:00


# Salvar Dataset Limpo

In [14]:
DIR_SAIDA = Path('dados_tratados')
DIR_SAIDA.mkdir(exist_ok=True)

for nome, df in dfs.items():
    df.to_csv(DIR_SAIDA / f'{nome}.csv', index=False)